# Lesson 10 - Agentes de IA em Produção

Nesta lição irá aprender **padrões de produção** para agentes de IA usando o Microsoft Agent Framework (Python). Cobrimos:

- **Observabilidade** — adicionar temporização e registo às interações do agente
- **Avaliação** — usar um agente avaliador para classificar a qualidade da resposta
- **Gestão de custos** — estratégias para otimização de tokens e seleção de modelo

O cenário é um **agente de viagens** que ajuda os utilizadores a planear viagens, com monitorização e avaliação sobrepostos.


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considerações para Produção

Mover agentes de IA de protótipos para produção requer uma atenção cuidadosa a três pilares:

1. **Observabilidade** — É necessário ter visibilidade sobre o que o agente está a fazer, quanto tempo demora e que ferramentas utiliza. Sem rastreamento e registo, depurar problemas em produção é quase impossível.

2. **Avaliação** — Verificações automatizadas de qualidade garantem que as respostas do agente se mantêm precisas, completas e úteis ao longo do tempo. Um agente avaliador pode pontuar as respostas com base em critérios definidos.

3. **Gestão de Custos** — O uso de tokens impacta diretamente o custo. Estratégias como otimização do prompt, seleção de modelo e armazenamento em cache ajudam a manter as despesas sob controlo sem sacrificar a qualidade.


## Construir um Agente Observável

Definimos ferramentas de viagem e envolvemos a chamada do agente com temporização para que possamos monitorizar a latência. Em produção, integraria com OpenTelemetry ou um backend de rastreio similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

Um padrão comum na produção é usar um segundo agente como **avaliador**. O avaliador pontua a resposta do agente principal com base em critérios predefinidos, como completude, precisão e utilidade.

Isto permite:
- Portões de qualidade automatizados antes das respostas chegarem aos utilizadores
- Detecção de regressão quando os prompts ou modelos mudam
- Monitorização contínua do desempenho do agente ao longo do tempo


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Estratégias de Gestão de Custos

Controlar os custos é fundamental para agentes de IA de produção. Aqui estão as estratégias principais:

| Estratégia | Descrição |
|---|---|
| **Otimização de prompts** | Mantenha as instruções do sistema concisas. Remova contexto redundante para reduzir os tokens de entrada. |
| **Seleção do modelo** | Utilize modelos mais pequenos e baratos (ex. GPT-4o-mini) para tarefas simples como classificação ou extração, e reserve modelos maiores para raciocínios complexos. |
| **Cache** | Guarde em cache os resultados das ferramentas e consultas frequentes para evitar chamadas redundantes à API. |
| **Orçamentos de tokens** | Defina limites de `max_tokens` para evitar respostas inesperadamente longas. |
| **Agrupamento** | Agrupe múltiplas consultas dos utilizadores numa única chamada à API sempre que possível. |

Na prática, uma abordagem por níveis funciona bem: direcione pedidos simples para um modelo rápido e económico e escale apenas consultas complexas para um modelo mais capaz.


## Resumo

Nesta lição aprendeu a:

1. **Adicionar observabilidade** às interações do agente com temporização e registos, estabelecendo as bases para rastreamento e monitorização.
2. **Avaliar respostas do agente** automaticamente utilizando um agente avaliador que pontua a completude, exatidão e utilidade.
3. **Gerir custos** através da otimização de prompts, seleção de modelos, cache e orçamentos de tokens.

Estes padrões de produção ajudam a garantir que os seus agentes de IA são fiáveis, mensuráveis e económicos em larga escala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
